In [ ]:
from embedder import Embedder

embed = Embedder()

In [ ]:
question = 'How does approximate nearest neighbor search work?'
question_embedding = embed.encode(question)

In [ ]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/07-sqlitesearch-vector" in path,
)

documents = [file.parse() for file in reader.read()]


In [ ]:
course_content=documents[0]["content"]

In [ ]:
# embedding course content
course = embed.encode(course_content)

In [ ]:
# Comparing the embeddings
course.dot(question_embedding)

In [ ]:
question_embedding.dot(course)

## Chunking course content

In [ ]:
# Loadding all the course content and embedding it
from ingest import load_data

documents = load_data()

In [ ]:
documents

In [ ]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [ ]:
texts = [chunk["content"] for chunk in chunks]

In [ ]:
from tqdm.auto import tqdm
import numpy as np

batch_size = 50
X = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    X.extend(batch_vectors)

X = np.array(X)

In [ ]:
scores = X.dot(question_embedding)

In [ ]:
# Finding highest scoring chunk
sorted_indices = np.argsort(scores)[::-1]

In [ ]:
idx = np.argmax(scores)

chunks[idx]

Vector search with minsearch

In [ ]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["content"])
vindex.fit(X, chunks)

In [ ]:
query = "What metric do we use to evaluate a search engine?"

In [ ]:
query_vector = embed.encode(query)

results = vindex.search(query_vector, num_results=5)

In [ ]:
results

## Text search vs vector search

In [ ]:
from ingest import build_index

index = build_index(chunks)

In [ ]:
query = "How do I store vectors in PostgreSQL?"

In [ ]:
query_vector = embed.encode(query)

vector_results = vindex.search(query_vector, num_results=5)
text_results = index.search(query, num_results=5)

In [ ]:
vector_results

In [ ]:
text_results

## Hybrid search

In [ ]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [ ]:
query = "How do I give the model access to tools?"

In [ ]:
query_vector = embed.encode(query)

vector_results = vindex.search(query_vector, num_results=5)
text_results = index.search(query, num_results=5)

In [ ]:
results = rrf([vector_results, text_results])

In [ ]:
results